In [1]:
import pandas as pd
from pathlib import Path

# Delhi raw data folder
data_path = Path("../data/raw/delhi/bus")

# Load the main GTFS files
routes = pd.read_csv(data_path / "routes.txt", dtype=str)
trips = pd.read_csv(data_path / "trips.txt", dtype=str)
stops = pd.read_csv(data_path / "stops.txt", dtype=str)
stop_times = pd.read_csv(data_path / "stop_times.txt", dtype=str)

# Display available columns
print("ROUTES:")
print(routes.columns.tolist())

print("\nTRIPS:")
print(trips.columns.tolist())

print("\nSTOPS:")
print(stops.columns.tolist())

print("\nSTOP TIMES:")
print(stop_times.columns.tolist())

ROUTES:
['agency_id', 'route_id', 'route_long_name', 'route_short_name', 'route_type']

TRIPS:
['route_id', 'service_id', 'trip_id', 'shape_id']

STOPS:
['stop_code', 'stop_id', 'stop_lat', 'stop_lon', 'stop_name', 'zone_id']

STOP TIMES:
['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence']


In [2]:
# 1. Count distinct trips for each route
trip_counts = (
    trips.groupby("route_id")["trip_id"]
    .nunique()
    .reset_index(name="total_trips")
)

# 2. Connect trips with their stops
trip_stops = trips[["route_id", "trip_id"]].merge(
    stop_times[["trip_id", "stop_id"]],
    on="trip_id",
    how="left"
)

# 3. Count distinct stops served by each route
stop_counts = (
    trip_stops.groupby("route_id")["stop_id"]
    .nunique()
    .reset_index(name="total_stops")
)

# 4. Combine both features
route_features = trip_counts.merge(
    stop_counts,
    on="route_id",
    how="left"
)

print(route_features.head(10))

  route_id  total_trips  total_stops
0        1           14           32
1       10           18           43
2     1000           12           72
3    10001           30           75
4    10004           30           44
5    10006           28           31
6    10007           48           41
7    10011            8           43
8    10014          104            4
9    10015           32           50


In [3]:
# Select route details
route_details = routes[["route_id", "route_short_name", "route_long_name"]]

# Combine route details with our features
route_features = route_details.merge(
    route_features,
    on="route_id",
    how="left"
)

print(route_features.head(10))

  route_id route_short_name route_long_name  total_trips  total_stops
0      142              NaN          828AUP           33           25
1    10001              NaN         971DOWN           30           75
2      362              NaN         961DOWN           16           94
3      274              NaN         819DOWN           20           43
4    10386              NaN     824STLDOWN2           23           31
5      233              NaN           919UP           51           41
6    10387              NaN    770ASTLDOWN2           51           21
7      118              NaN     817NSTLDOWN           62           11
8    10388              NaN         73ADOWN           51           26
9       67              NaN        112STLUP           10           16


In [4]:
# Display the first 10 scheduled arrival and departure times
print(stop_times[["arrival_time", "departure_time"]].head(10))

# Check for missing time values
print("\nMissing arrival times:", stop_times["arrival_time"].isna().sum())
print("Missing departure times:", stop_times["departure_time"].isna().sum())

# Check whether times exceed 24 hours
print("\nTimes beyond 24 hours:")
print(stop_times[
    stop_times["arrival_time"].str.startswith("24:", na=False)
    | stop_times["arrival_time"].str.startswith("25:", na=False)
][["arrival_time", "departure_time"]].head())

  arrival_time departure_time
0     12:50:00       12:50:00
1     12:51:11       12:51:11
2     12:52:49       12:52:49
3     12:53:54       12:53:54
4     12:55:44       12:55:44
5     12:57:10       12:57:10
6     12:58:43       12:58:43
7     13:00:46       13:00:46
8     13:04:51       13:04:51
9     13:06:08       13:06:08

Missing arrival times: 0
Missing departure times: 0

Times beyond 24 hours:
      arrival_time departure_time
15491     24:02:04       24:02:04
15492     24:05:00       24:05:00
15493     24:07:41       24:07:41
16163     24:01:45       24:01:45
16164     24:04:48       24:04:48


In [5]:
# Convert GTFS time into total seconds, preserving hours beyond 24
def gtfs_time_to_hour(time_value):
    if pd.isna(time_value):
        return None

    try:
        hours, minutes, seconds = map(int, time_value.split(":"))
        return hours + minutes / 60 + seconds / 3600
    except (ValueError, AttributeError):
        return None


# Extract arrival and departure hours
stop_times["arrival_hour"] = stop_times["arrival_time"].apply(gtfs_time_to_hour)
stop_times["departure_hour"] = stop_times["departure_time"].apply(gtfs_time_to_hour)


# Classify time periods
def get_time_period(hour):
    if pd.isna(hour):
        return "Unknown"
    elif 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"


stop_times["arrival_period"] = stop_times["arrival_hour"].apply(get_time_period)
stop_times["departure_period"] = stop_times["departure_hour"].apply(get_time_period)


# Display results
print(stop_times[
    ["arrival_time", "arrival_hour", "arrival_period",
     "departure_time", "departure_hour", "departure_period"]
].head(10))

  arrival_time  arrival_hour arrival_period departure_time  departure_hour  \
0     12:50:00     12.833333      Afternoon       12:50:00       12.833333   
1     12:51:11     12.853056      Afternoon       12:51:11       12.853056   
2     12:52:49     12.880278      Afternoon       12:52:49       12.880278   
3     12:53:54     12.898333      Afternoon       12:53:54       12.898333   
4     12:55:44     12.928889      Afternoon       12:55:44       12.928889   
5     12:57:10     12.952778      Afternoon       12:57:10       12.952778   
6     12:58:43     12.978611      Afternoon       12:58:43       12.978611   
7     13:00:46     13.012778      Afternoon       13:00:46       13.012778   
8     13:04:51     13.080833      Afternoon       13:04:51       13.080833   
9     13:06:08     13.102222      Afternoon       13:06:08       13.102222   

  departure_period  
0        Afternoon  
1        Afternoon  
2        Afternoon  
3        Afternoon  
4        Afternoon  
5        Aftern

In [6]:
# Check which GTFS files are available
print("Available GTFS files:")

for filename in [
    "calendar.txt",
    "calendar_dates.txt",
    "shapes.txt",
    "frequencies.txt"
]:
    file_path = data_path / filename
    print(filename, ":", "Available" if file_path.exists() else "Not available")


# Check stop_times columns
print("\nStop times columns:")
print(stop_times.columns.tolist())

Available GTFS files:
calendar.txt : Available
calendar_dates.txt : Not available
shapes.txt : Not available
frequencies.txt : Not available

Stop times columns:
['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence', 'arrival_hour', 'departure_hour', 'arrival_period', 'departure_period']


In [7]:
# Load the service calendar
calendar = pd.read_csv(data_path / "calendar.txt", dtype=str)

# Display its columns and first 10 rows
print("Calendar columns:")
print(calendar.columns.tolist())

print("\nCalendar data:")
print(calendar.head(10))

Calendar columns:
['start_date', 'end_date', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday', 'service_id']

Calendar data:
  start_date  end_date monday tuesday wednesday thursday friday saturday  \
0   20240101  20270101      1       1         1        1      1        1   

  sunday service_id  
0      1          1  


In [8]:
# Get the first departure time of each trip
first_departures = (
    stop_times.sort_values("stop_sequence")
    .drop_duplicates("trip_id")
)

# Connect each trip with its route
trip_hours = first_departures.merge(
    trips[["trip_id", "route_id"]],
    on="trip_id",
    how="left"
)

# Count distinct trips by route and departure hour
hourly_trips = (
    trip_hours.groupby(["route_id", "departure_hour"])["trip_id"]
    .nunique()
    .reset_index(name="trip_count")
)

print(hourly_trips.head(10))

  route_id  departure_hour  trip_count
0        1        0.033333           1
1        1       19.616667           1
2        1       20.233333           1
3        1       20.850000           1
4        1       21.033333           1
5        1       21.466667           1
6        1       21.666667           1
7        1       22.066667           1
8        1       22.083333           1
9        1       22.466667           1


In [9]:
# Combine route details and trip/stop features
route_profile = route_features.copy()

# Add total operating days
weekday_columns = [
    "monday", "tuesday", "wednesday", "thursday",
    "friday", "saturday", "sunday"
]

calendar["operating_days"] = calendar[weekday_columns].astype(int).sum(axis=1)

print("Operating days:")
print(calendar[["service_id", "operating_days"]])

# Save route profile
output_path = Path("../data/processed/delhi")
output_path.mkdir(parents=True, exist_ok=True)

route_profile.to_csv(output_path / "route_profile.csv", index=False)

# Save hourly trip counts
hourly_trips.to_csv(output_path / "hourly_trip_counts.csv", index=False)

print("\nDelhi route profile saved successfully!")
print(route_profile.head())

Operating days:
  service_id  operating_days
0          1               7

Delhi route profile saved successfully!
  route_id route_short_name route_long_name  total_trips  total_stops
0      142              NaN          828AUP           33           25
1    10001              NaN         971DOWN           30           75
2      362              NaN         961DOWN           16           94
3      274              NaN         819DOWN           20           43
4    10386              NaN     824STLDOWN2           23           31


In [10]:
print("Missing values:")
print(route_profile.isnull().sum())

print("\nDuplicate route IDs:")
print(route_profile["route_id"].duplicated().sum())

print("\nDuplicate trip IDs:")
print(trips["trip_id"].duplicated().sum())

print("\nInvalid or missing arrival hours:")
print(stop_times["arrival_hour"].isna().sum())

print("\nInvalid or missing departure hours:")
print(stop_times["departure_hour"].isna().sum())

Missing values:
route_id               0
route_short_name    2403
route_long_name        0
total_trips            0
total_stops            0
dtype: int64

Duplicate route IDs:
0

Duplicate trip IDs:
0

Invalid or missing arrival hours:
0

Invalid or missing departure hours:
0


In [11]:
feature_dictionary = pd.DataFrame([
    ["route_id", "Unique route identifier", "routes.txt", "Original field", "Categorical"],
    ["route_long_name", "Route name", "routes.txt", "Original field", "Categorical"],
    ["total_trips", "Distinct scheduled trips per route", "trips.txt", "Group by route and count unique trips", "Numerical"],
    ["total_stops", "Distinct stops served per route", "trips.txt + stop_times.txt", "Join and count unique stops", "Numerical"],
    ["arrival_hour", "Scheduled arrival hour", "stop_times.txt", "Extract from GTFS arrival time", "Numerical"],
    ["departure_hour", "Scheduled departure hour", "stop_times.txt", "Extract from GTFS departure time", "Numerical"],
    ["arrival_period", "Arrival time category", "stop_times.txt", "Morning/Afternoon/Evening/Night", "Categorical"],
    ["departure_period", "Departure time category", "stop_times.txt", "Morning/Afternoon/Evening/Night", "Categorical"],
    ["trip_count", "Distinct trips departing in an hour for a route", "trips.txt + stop_times.txt", "Count first departures by route and hour", "Numerical"]
], columns=["Feature", "Meaning", "Source", "Calculation", "Data_Type"])

feature_dictionary.to_csv(
    output_path / "feature_dictionary.csv",
    index=False
)

print("Feature dictionary saved successfully!")

Feature dictionary saved successfully!


In [12]:
# Extract minutes
def gtfs_minute(t):
    try:
        h, m, s = map(int, t.split(":"))
        return m
    except:
        return None

stop_times["arrival_minute"] = stop_times["arrival_time"].apply(gtfs_minute)
stop_times["departure_minute"] = stop_times["departure_time"].apply(gtfs_minute)

# Correct hourly grouping
stop_times["stop_sequence"] = pd.to_numeric(
    stop_times["stop_sequence"], errors="coerce"
)

first_departures = (
    stop_times.sort_values("stop_sequence")
    .drop_duplicates("trip_id")
)

first_departures["departure_service_hour"] = (
    first_departures["departure_hour"].apply(
        lambda h: int(h) if pd.notna(h) else None
    )
)

trip_hours = first_departures.merge(
    trips[["trip_id", "route_id"]],
    on="trip_id",
    how="left"
)

hourly_trips = (
    trip_hours.groupby(["route_id", "departure_service_hour"])["trip_id"]
    .nunique()
    .reset_index(name="trip_count")
)

# Save corrected files
output_path = Path("../data/processed/delhi")
hourly_trips.to_csv(output_path / "hourly_trip_counts.csv", index=False)
stop_times.to_csv(output_path / "time_features.csv", index=False)

print("Delhi updated successfully!")

Delhi updated successfully!
